# PredictiveSense AI 🚀
## Day 9 - LSTM Deep Learning Model
### Time Series RUL Prediction using Neural Network

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load ML-ready dataset
df = pd.read_csv(r'C:\Users\vipin nagar\OneDrive\Desktop\Internship 2026\PredictiveSense-AI\data\processed\train_final.csv')

print("✅ Data loaded!")
print(f"Shape: {df.shape}")

✅ Data loaded!
Shape: (20631, 21)


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# RUL Capping
RUL_CAP = 125
df['RUL'] = df['RUL'].clip(upper=RUL_CAP)

# Features & Target
feature_cols = [col for col in df.columns 
                if col not in ['RUL', 'unit_id', 'cycle']]
X = df[feature_cols].values
y = df['RUL'].values

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("✅ Data prepared!")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Features: {len(feature_cols)}")

✅ Data prepared!
X_train: (16504, 18)
X_test : (4127, 18)
Features: 18


In [3]:
# LSTM ko 3D input chahiye — (samples, timesteps, features)
# Abhi data 2D hai — (samples, features)
# Hum timestep = 1 rakhenge — simple approach

X_train_lstm = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test_lstm  = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

print("✅ Data reshaped for LSTM!")
print(f"X_train_lstm shape: {X_train_lstm.shape}")
print(f"X_test_lstm shape : {X_test_lstm.shape}")
print(f"\nFormat: (samples, timesteps, features)")

✅ Data reshaped for LSTM!
X_train_lstm shape: (16504, 1, 18)
X_test_lstm shape : (4127, 1, 18)

Format: (samples, timesteps, features)


In [4]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size=18, hidden_size=64, num_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm    = nn.LSTM(input_size, hidden_size, 
                               num_layers, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc1     = nn.Linear(hidden_size, 32)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(32, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out     = self.dropout(out[:, -1, :])
        out     = self.relu(self.fc1(out))
        out     = self.fc2(out)
        return out

model = LSTMModel()
print("✅ LSTM Model ready!")
print(model)

✅ LSTM Model ready!
LSTMModel(
  (lstm): LSTM(18, 64, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)


In [5]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Data ko PyTorch tensors mein convert karo
X_train_t = torch.FloatTensor(X_train_lstm)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
X_test_t  = torch.FloatTensor(X_test_lstm)
y_test_t  = torch.FloatTensor(y_test).reshape(-1, 1)

# DataLoader
dataset    = TensorDataset(X_train_t, y_train_t)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Loss & Optimizer
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training Loop
print("⏳ Training LSTM...")
epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(dataloader):.4f}")

print("✅ LSTM Training complete!")

⏳ Training LSTM...
Epoch 5/20 — Loss: 906.8880
Epoch 10/20 — Loss: 436.1489
Epoch 15/20 — Loss: 428.5737
Epoch 20/20 — Loss: 413.8046
✅ LSTM Training complete!


In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Evaluation
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t)
    
y_pred = y_pred_t.numpy().flatten()
y_true = y_test_t.numpy().flatten()

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print("=== LSTM RESULTS ===")
print(f"RMSE : {rmse:.2f} cycles")
print(f"MAE  : {mae:.2f} cycles")
print(f"R2   : {r2:.4f}")

print("\n=== COMPARISON ===")
print(f"Random Forest R2 : 0.7949")
print(f"LSTM R2          : {r2:.4f}")

=== LSTM RESULTS ===
RMSE : 19.57 cycles
MAE  : 14.88 cycles
R2   : 0.7742

=== COMPARISON ===
Random Forest R2 : 0.7949
LSTM R2          : 0.7742


In [7]:
import torch
import os

# Save LSTM model
torch.save(model.state_dict(), 
    r'C:\Users\vipin nagar\OneDrive\Desktop\Internship 2026\PredictiveSense-AI\models\lstm_model.pth')

print("✅ LSTM Model saved!")
print("\n=== FINAL MODEL COMPARISON ===")
print(f"Random Forest — R2: 0.7949 | RMSE: 18.66")
print(f"LSTM          — R2: 0.7675 | RMSE: 19.86")
print(f"\n🏆 Best Model: Random Forest")
print(f"📌 LSTM shows competitive performance with only 20 epochs")

✅ LSTM Model saved!

=== FINAL MODEL COMPARISON ===
Random Forest — R2: 0.7949 | RMSE: 18.66
LSTM          — R2: 0.7675 | RMSE: 19.86

🏆 Best Model: Random Forest
📌 LSTM shows competitive performance with only 20 epochs
